In [21]:
import numpy as np
from scipy.spatial import cKDTree

In [22]:
class Source:
    def __init__(self, Y=10000):
        self.Y = Y

    @staticmethod
    def _generate_directions_array(n):
        """Векторная генерация n изотропных направлений."""
        phi = np.random.uniform(0, 2 * np.pi, n)
        cos_theta = np.random.uniform(-1, 1, n)

        sin_theta = np.sqrt(1 - cos_theta**2)

        return np.column_stack((
            sin_theta * np.cos(phi),
            sin_theta * np.sin(phi),
            cos_theta
        ))

    def generate_directions_array(self, E):
        """Генерирует массив направлений всех фотонов события."""
        if E < 0:
            raise ValueError("Энергия не может быть отрицательной.")

        n_photons = np.random.poisson(self.Y * E)

        return self._generate_directions_array(n_photons)

    def generate_directions(self, E, batch_size=100):
        """Генератор, возвращающий направления порциями по batch_size."""
        if E < 0:
            raise ValueError("Энергия не может быть отрицательной.")
        if batch_size <= 0:
            raise ValueError("Размер порции должен быть положительным.")

        n_photons = np.random.poisson(self.Y * E)

        for start in range(0, n_photons, batch_size):
            n = min(batch_size, n_photons - start)
            yield self._generate_directions_array(n)

In [23]:
class Detector:
    def __init__(self, R=15, N=2000, r=0.1, a=0.5):
        self.R = R
        self.N = N
        self.r = r
        self.a = a

        # Равномерное распределение ФЭУ на сфере (спираль Фибоначчи)
        indices = np.arange(N)
        z = 1 - 2 * (indices + 0.5) / N
        phi = np.pi * (3 - np.sqrt(5)) * indices
        rho = np.sqrt(1 - z**2)

        x = rho * np.cos(phi)
        y = rho * np.sin(phi)

        self.pm_positions = R * np.column_stack((x, y, z))

        # Построение KD-дерева по положениям ФЭУ
        self.pm_tree = cKDTree(self.pm_positions)

        # Радиус фотокатода (длина хорды на сфере)
        self.pm_chord_radius = 2 * R * np.sin(r / (2 * R))

    def propagate_photons(self, directions, source_position, attenuation_length=None):
        """
        Прослеживает пути фотонов от источника до поверхности сферы 
        и проверяет попадание в ФЭУ с помощью KD-дерева.
        """
        directions = np.asarray(directions)
        source_position = np.asarray(source_position)

        # Расстояние от источника до поверхности детектора (t)
        dot = directions @ source_position
        t = -dot + np.sqrt(dot**2 + self.R**2 - np.dot(source_position, source_position))

        # Точки попадания фотонов на сферу
        hit_points = source_position + t[:, None] * directions

        # Поиск ближайшего ФЭУ с помощью KD-дерева (k=1)
        distances, _ = self.pm_tree.query(hit_points, k=1)

        # Проверка, попал ли фотон на фотокатод
        hits = distances <= self.pm_chord_radius

        # Учёт поглощения
        if attenuation_length is not None:
            if attenuation_length <= 0:
                raise ValueError("Длина поглощения должна быть положительной.")
            survival_probability = np.exp(-t / attenuation_length)
            survived = np.random.random(len(t)) < survival_probability
            hits &= survived

        return hits, t

In [24]:
def find_hits_naive(detector, hit_points, chunk_size=256):
    hits = np.zeros(len(hit_points), dtype=bool)

    pm_positions = detector.pm_positions / detector.R
    cos_limit = np.cos(detector.r / detector.R)

    for start in range(0, len(hit_points), chunk_size):
        end = min(start + chunk_size, len(hit_points))

        points = hit_points[start:end] / detector.R

        cos_angles = points @ pm_positions.T

        hits[start:end] = np.any(
            cos_angles >= cos_limit,
            axis=1
        )

    return hits

In [25]:
detector = Detector(R=15, N=2000, r=0.1, a=0.5)
source = Source(Y=10000)

n_photons = 10000

directions = source._generate_directions_array(n_photons)

# Источник находится в центре детектора
hit_points = detector.R * directions

# KD-дерево
distances, _ = detector.pm_tree.query(hit_points, k=1)
hits_tree = distances <= detector.pm_chord_radius

# Проверка всех ФЭУ
hits_naive = find_hits_naive(detector, hit_points)

print("Количество фотонов:", n_photons)
print("Попадания (KD-дерево):", np.count_nonzero(hits_tree))
print("Попадания (полный перебор):", np.count_nonzero(hits_naive))
print("Количество различий:", np.count_nonzero(hits_tree != hits_naive))

Количество фотонов: 10000
Попадания (KD-дерево): 224
Попадания (полный перебор): 224
Количество различий: 0


In [26]:
print("Замер времени для KD-дерева:")
%timeit detector.pm_tree.query(hit_points, k=1)

print("\nЗамер времени для полного перебора:")
%timeit find_hits_naive(detector, hit_points)

Замер времени для KD-дерева:
4.66 ms ± 80.6 μs per loop (mean ± std. dev. of 7 runs, 100 loops each)

Замер времени для полного перебора:
52.9 ms ± 1.99 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


Использование KD-дерева дало ускорение в ~12 раз по сравнению с полным перебором, так как сложность поиска снизилась с O(N) до O(log⁡N)